In [26]:
import json
import os
import subprocess
import itertools
from datetime import datetime
import pandas as pd
from pathlib import Path
import sys

In [27]:
env = os.environ.copy()
env['MKL_SERVICE_FORCE_INTEL'] = '1'
env['MKL_THREADING_LAYER'] = 'GNU'
env['OMP_NUM_THREADS'] = '1'

In [28]:

# Define parameter grids - modify these as needed
parameter_grid = {
    "n_total": [5000],
    "n_finetune": [2500],
    "model_name": ["bert-base-uncased"],
    "max_length": [256],
    "num_labels": [4],
    "batch_size": [16],
    "learning_rate": [2e-4, 2e-5],
    "num_epochs": [4,5],
    "K": [15],
    "lambda_min": [0.05],
    "lambda_max": [0.95],
    "interpolation": ["model_baseline", "linear","quad"],
}

# Or define specific combinations
specific_configs = [
    {
        "n_total": 5000,
        "n_finetune": 2500,
        "model_name": "bert-base-uncased",
        "max_length": 256,
        "num_labels": 4,
        "batch_size": 16,
        "learning_rate": 2e-5,
        "num_epochs": 1,
        "K": 3,
        "lambda_min": 0.05,
        "lambda_max": 0.95,
        "interpolation": "model_baseline"
    },
    {
        "n_total": 3000,
        "n_finetune": 1500,
        "model_name": "distilbert-base-uncased",
        "max_length": 128,
        "num_labels": 4,
        "batch_size": 32,
        "learning_rate": 5e-5,
        "num_epochs": 3,
        "K": 2,
        "lambda_min": 0.1,
        "lambda_max": 0.9,
        "interpolation": "linear"
    }
]

def generate_all_combinations(param_grid):
    keys = param_grid.keys()
    values = param_grid.values()
    combinations = []
    
    for combination in itertools.product(*values):
        config = dict(zip(keys, combination))
        combinations.append(config)
    
    return combinations

def create_config_file(config, experiment_path):
    config['experiment_name'] = experiment_path
    config_path = experiment_path + ".json"
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=4)
    print(f"Created config file: {config_path}")

def run_experiment(config, experiment_name):
    try:
        # Create config file
        create_config_file(config, f"{experiment_name}")
        
        print(f"\nRunning experiment: {experiment_name}")
        print(f"Config: {config}")
        
        # result = subprocess.run([sys.executable, "agnews_sample_hacking_last_layer.py", f"{experiment_name}.json"], capture_output=True, text=True,shell=True)
        result = subprocess.call([sys.executable, "agnews_sample_hacking_last_layer.py", f"{experiment_name}.json"],env=env)
        
        if result == 0:
            print(f"✅ Experiment {experiment_name} completed successfully")
        else:
            print(f"❌ Experiment {experiment_name} failed")
            # print("Error:", result.stderr)
        
        return {
            "experiment_name": experiment_name,
            "config": config,
            "success": result == 0,
            # "stdout": result.stdout,
            # "stderr": result.stderr
        }
        
    except Exception as e:
        print(f"❌ Exception in {experiment_name}: {str(e)}")
        return {
            "experiment_name": experiment_name,
            "config": config,
            "success": False,
            "error": str(e)
        }

In [29]:
# Choose experiment mode
USE_GRID_SEARCH = False # Set to True for grid search, False for specific configs

if USE_GRID_SEARCH:
    configs_to_run = generate_all_combinations(parameter_grid )
else:
    configs_to_run = specific_configs

print(f"Total configurations to run: {len(configs_to_run)}")

Total configurations to run: 2


In [30]:
# When a parent process starts a child process via subprocess, the two are separate entities with their own memory space.
# The parent process is not notified in real-time about the filesystem modifications the child process is making.

In [ ]:
# Run all experiments
results = []
for i, config in enumerate(configs_to_run):
    experiment_name = f"{config['num_epochs']}_{config['learning_rate']}_{config['interpolation']}"
    result = run_experiment(config, experiment_name)
    results.append(result)

# Summary
successful = sum(1 for r in results if r["success"])
print(f"\n{'='*50}")
print(f"EXPERIMENT SUMMARY")
print(f"{'='*50}")
print(f"Total experiments: {len(results)}")
print(f"Successful: {successful}")
print(f"Failed: {len(results) - successful}")